# 02 — Text Preprocessing

Iterate on the cleaning pipeline that ships in `app/ml/preprocessing.py`.
We test each transformation in isolation before locking it into the production code.

In [ ]:
import sys
sys.path.append('..')
from app.ml.preprocessing import preprocess
import pandas as pd
df = pd.read_csv('../data/raw/tickets.csv')

## Before / after on a sample

In [ ]:
sample = df.sample(5, random_state=7)
for _, row in sample.iterrows():
    print('RAW :', row['text'][:160])
    print('CLEAN:', preprocess(row['text'])[:160])
    print('-' * 80)

## Vocabulary shrinkage check

Sanity-check that preprocessing collapses the vocabulary meaningfully (lemmatisation + stopword removal).

In [ ]:
raw_vocab = set(' '.join(df['text']).lower().split())
clean_vocab = set(' '.join(df['text'].map(preprocess)).split())
print(f'raw   vocab: {len(raw_vocab):,}')
print(f'clean vocab: {len(clean_vocab):,}')
print(f'reduction  : {1 - len(clean_vocab)/len(raw_vocab):.1%}')

## Notes

- Stripping URLs / emails / order-IDs **before** tokenisation matters — TF-IDF was over-weighting `order_id` patterns as a leakage feature in early runs.
- WordNet lemmatisation (default POS = noun) gave a ~3 % macro-F1 lift vs Porter stemming; we kept it.
- Kept basic punctuation removal but did NOT lowercase currency symbols (`$`, `₹`) — useful for `Billing` class.